In [ ]:
from enum import auto

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from helpers.pathlib import get_path_site_checkpoint

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow, _StrEnum
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
CSV_PREFIX = "221103-230125"

In [ ]:
class FieldTemp(_StrEnum):
    DEVIATION_FROM_GROUP_DOC_HEIGHT_MEAN = auto()
    DEVIATION_FROM_GROUP_BR_VIEWHEIGHT_MEAN = auto()
    GROUP_DOC_HEIGHT_MEAN = auto()
    GROUP_BR_VIEWHEIGHT_MEAN = auto()
    GROUP_MEMBER_COUNT = auto()

In [ ]:
# Read data from CHECKPOINT 2
df_afla = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, THE_19TH.name))

In [ ]:
def calculate_group_means(df):
    assert df[FieldSnowplow.DOC_HEIGHT].isna().sum() == 0
    assert df[FieldSnowplow.BR_VIEWHEIGHT].isna().sum() == 0
    df_grouped = (
        df.reset_index()
        .groupby(FieldNew.EVENT_PARENT_ID)
        .aggregate(
            **{
                FieldTemp.GROUP_DOC_HEIGHT_MEAN: pd.NamedAgg(column=FieldSnowplow.DOC_HEIGHT, aggfunc="mean"),
                FieldTemp.GROUP_BR_VIEWHEIGHT_MEAN: pd.NamedAgg(column=FieldSnowplow.BR_VIEWHEIGHT, aggfunc="mean"),
                FieldTemp.GROUP_MEMBER_COUNT: pd.NamedAgg(column=FieldSnowplow.EVENT_ID, aggfunc="size"),
            }
        )
    )
    return df.merge(df_grouped, how="left", left_on=FieldNew.EVENT_PARENT_ID, right_index=True)


df_afla = calculate_group_means(df_afla)
df_dfp = calculate_group_means(df_dfp)
df_ov = calculate_group_means(df_ov)
df_19 = calculate_group_means(df_19)

In [ ]:
df_afla[FieldTemp.GROUP_MEMBER_COUNT].describe()

In [ ]:
df_dfp[FieldTemp.GROUP_MEMBER_COUNT].describe()

In [ ]:
df_ov[FieldTemp.GROUP_MEMBER_COUNT].describe()

In [ ]:
df_19[FieldTemp.GROUP_MEMBER_COUNT].describe()

In [ ]:
def calculate_residuals(df, group_member_count_cutoff: int = 1):
    return df.query(f"{FieldTemp.GROUP_MEMBER_COUNT} > {group_member_count_cutoff}").assign(
        **{
            FieldTemp.DEVIATION_FROM_GROUP_DOC_HEIGHT_MEAN: lambda x: x[FieldTemp.GROUP_DOC_HEIGHT_MEAN]
            - x[FieldSnowplow.DOC_HEIGHT],
            FieldTemp.DEVIATION_FROM_GROUP_BR_VIEWHEIGHT_MEAN: lambda x: x[FieldTemp.GROUP_BR_VIEWHEIGHT_MEAN]
            - x[FieldSnowplow.BR_VIEWHEIGHT],
        }
    )


df_afla = calculate_residuals(df_afla)
df_dfp = calculate_residuals(df_dfp)
df_ov = calculate_residuals(df_ov)
df_19 = calculate_residuals(df_19)

## Plot `doc_height` residuals

In [ ]:
def plot_doc_height(df, site_name, ax):
    sns.histplot(df, x=FieldTemp.DEVIATION_FROM_GROUP_DOC_HEIGHT_MEAN, ax=ax, bins=20, kde=True)
    ax.xaxis.set_label_text("Pixels")


_, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
plot_doc_height(df_afla, AFRO_LA.name, axes[0, 0])
plot_doc_height(df_dfp, DALLAS_FREE_PRESS.name, axes[0, 1])
plot_doc_height(df_ov, OPEN_VALLEJO.name, axes[1, 0])
plot_doc_height(df_19, THE_19TH.name, axes[1, 1])
plt.show()

## Plot `br_viewheight` residuals

In [ ]:
def plot_br_viewheight(df, site_name, ax):
    sns.histplot(df, x=FieldTemp.DEVIATION_FROM_GROUP_BR_VIEWHEIGHT_MEAN, ax=ax, bins=10, kde=True)
    ax.xaxis.set_label_text("Pixels")


_, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
plot_br_viewheight(df_afla, AFRO_LA.name, axes[0, 0])
plot_br_viewheight(df_dfp, DALLAS_FREE_PRESS.name, axes[0, 1])
plot_br_viewheight(df_ov, OPEN_VALLEJO.name, axes[1, 0])
plot_br_viewheight(df_19, THE_19TH.name, axes[1, 1])
plt.show()